# Demand Prediction Notebook

## Data And Libraries Import

In [ ]:
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns


import os
from dotenv import load_dotenv
from pathlib import Path


import warnings
warnings.filterwarnings('ignore')

In [4]:
load_dotenv()
out = os.getenv("OUTPUT_DIR")

In [5]:
lead = pd.read_csv(os.path.join(out, "lead_by_year.csv"), parse_dates=['festival_date'])
groups = pd.read_csv(os.path.join(out, "state_groups.csv"))

lead['year'] = lead['festival_date'].dt.year
lead.head()

,festival,state,festival_date,lead_weeks,year
0,Bihu (Rongali),Andhra Pradesh,2020-04-14,0.0,2020
1,Bihu (Rongali),Andhra Pradesh,2021-04-14,0.0,2021
2,Bihu (Rongali),Andhra Pradesh,2022-04-14,1.0,2022
3,Bihu (Rongali),Andhra Pradesh,2023-04-14,0.0,2023
4,Bihu (Rongali),Andhra Pradesh,2024-04-14,NaN,2024


In [6]:
train = lead[lead['year'] <= 2024]
test = lead[lead['year'] == 2025]

pred = (train.groupby(['festival', 'state'])['lead_weeks']
             .median()
             .reset_index(name='predicted'))

check = test.merge(pred, on=['festival', 'state'])
check = check.rename(columns={'lead_weeks': 'actual'})
check[['festival', 'state', 'actual', 'predicted']].head()

,festival,state,actual,predicted
0,Bihu (Rongali),Andhra Pradesh,0.0,0.0
1,Bihu (Rongali),Assam,1.0,1.0
2,Bihu (Rongali),Bihar,0.0,0.0
3,Bihu (Rongali),Delhi,1.0,0.0
4,Bihu (Rongali),Gujarat,1.0,0.0


## Analysis

In [7]:
check = check.dropna(subset=['actual', 'predicted'])
check['error'] = (check['actual'] - check['predicted']).abs()

check['error'].mean()

np.float64(1.32)

In [8]:
base = (train.groupby('festival')['lead_weeks'].median()
             .reset_index(name='baseline'))

check = check.merge(base, on='festival')
(check['actual'] - check['baseline']).abs().mean()

np.float64(1.7733333333333334)

In [9]:
check.groupby('festival')['error'].mean().round(2)

festival
Bihu (Rongali)         0.60
Christmas              0.47
Diwali                 0.93
Durga Puja/Navratri    2.80
Eid al-Fitr            1.33
Ganesh Chaturthi       1.33
Holi                   2.07
Onam                   1.13
Pongal                 1.00
Raksha Bandhan         1.53
Name: error, dtype: float64

In [10]:
dp = check[check['festival'] == 'Durga Puja/Navratri']
dp[['state', 'actual', 'predicted', 'error']].sort_values('error', ascending=False)

,state,actual,predicted,error
51,Kerala,8.0,3.0,5.0
58,Uttar Pradesh,7.0,2.0,5.0
48,Delhi,7.0,3.0,4.0
54,Punjab,7.0,3.0,4.0
55,Rajasthan,8.0,4.0,4.0
53,Odisha,11.0,7.0,4.0
49,Gujarat,10.0,7.0,3.0
50,Karnataka,7.0,5.0,2.0
45,Andhra Pradesh,6.0,4.0,2.0
46,Assam,11.0,9.0,2.0


In [11]:
dp_all = lead[lead['festival'] == 'Durga Puja/Navratri']
dp_all.groupby('year').agg(lead=('lead_weeks', 'median'), date=('festival_date', 'first'))

,lead,date
year,,
2020,3.0,2020-10-22
2021,4.0,2021-10-12
2022,5.0,2022-10-02
2023,5.0,2023-10-21
2024,5.0,2024-10-10
2025,7.0,2025-09-29


In [12]:
train3 = lead[lead['year'].between(2022, 2024)]

pred3 = (train3.groupby(['festival', 'state'])['lead_weeks']
               .median().reset_index(name='pred3'))

check = check.merge(pred3, on=['festival', 'state'])
(check['actual'] - check['pred3']).abs().mean()

np.float64(1.17)

In [13]:
rows = []
for y in [2023, 2024, 2025]:
    t = lead[lead['year'] == y].dropna(subset=['lead_weeks'])
    for name, tr in [('last3', lead[lead['year'].between(y-3, y-1)]),
                     ('all_past', lead[lead['year'] < y])]:
        p = tr.groupby(['festival', 'state'])['lead_weeks'].median().reset_index(name='p')
        m = t.merge(p, on=['festival', 'state']).dropna(subset=['p'])
        rows.append((y, name, (m['lead_weeks'] - m['p']).abs().mean()))

pd.DataFrame(rows, columns=['year', 'method', 'MAE']).pivot(index='year', columns='method', values='MAE').round(2)

method,all_past,last3
year,,
2023,1.37,1.37
2024,1.09,1.10
2025,1.32,1.17


In [15]:
groups.columns

Index(['Unnamed: 0', 'group_name'], dtype='object')

In [16]:
groups = pd.read_csv(os.path.join(out, "state_groups.csv"), index_col=0)
groups = groups.rename_axis('state').reset_index()

In [17]:
state_med = (lead.groupby(['festival', 'state'])['lead_weeks']
                 .median().reset_index()
                 .merge(groups, on='state'))

plan = (state_med.pivot_table(index='festival', columns='group_name',
                              values='lead_weeks', aggfunc='mean')
                 .drop(['Bihu (Rongali)', 'Eid al-Fitr'])
                 .round(1))
plan

group_name,East,North,South,West & Deccan
festival,,,,
Christmas,4.2,4.6,5.7,5.8
Diwali,6.1,10.0,8.0,8.9
Durga Puja/Navratri,8.4,3.2,3.8,5.6
Ganesh Chaturthi,4.4,3.9,5.0,7.2
Holi,6.5,9.1,2.2,7.6
Onam,3.2,4.0,6.2,5.5
Pongal,1.9,2.5,3.8,3.5
Raksha Bandhan,6.0,8.2,5.2,7.4
